<div style="padding: 10px; border: 1px solid #ced4da; background-color: #e9ecef; border-radius: 5px;">

### Комментарии исполнителя:

- я начал с кода, данного на напрактике по RAG

- код модифицировал, чтобы обе модели (эмбедер и реранкер) помещались на GPU. Для этого написан код, который держит на GPU в моменте только ту модель, которая сейчас востребована

- мой бейзлайн состоял из маленького эмбедера и большого реранкера. Профилирование этапов поиска показало, что основное время уходило на ранжирование большой моделью на втором этапе поиска - переранжирование кандидатов из индекса. Тогда я решил попробовать взять эмбедер побольше, чтобы сократить кол-во кандидатов для переранжирования. Но тут сделал ошибку: замерил MRR для поисковых выдач разной длины и неправильно их интерпретировал. Сначала я решил, что `MRR@100 < 0.91` для первого этапа поиска гарантированно не даст `MRR@5 >= 0.91` на втором этапе. Тут я не учёл что MRR учитывает позицию, а не отражает покрытие выдачи правильными ответами.

- попробовав большой эмбедер получилось, ценой довольно долгого построения индекса, получить `MRR@5 >= 0.91` даже на первом этапе (без переранжирования). В целом это решает поставленную задачу, но я решил поискать быстрое решение с использованием изначально задуманной архитектуры с двумя стадиями поиска.

- далее я попробовал base-эмбедер и base-реранкер (специальный cross-encoder, а не генеративный Qwen, как было на практике). По графику покрытия выдачи правильными ответами я увидел, что даже 5 кандидатов на первом этапе (поиск по индексу) может быть достаточно для получения целевого качества: `Coverage@5: 0.938 | MRR@5: 0.89`. Т.е. поиск по индексу дает MRR@5 < 0.91, но покрытие > 0.91 => хороший реранкер может поднять правильный ответ выше и улучшить score. Так и вышло. MRR@5 второго этапа (переранжирование) получил `MRR@5=0.9192`. А весь тестовая выборка из 1000 запросов была обработана за 2 минуты.

- что ещё можно сделать чтобы ускорить решение:
  - как будто самая выгодная стратегия: как можно меньше вызывать реранкер. Переранжирование - самый медленный этап. Поэтому можно:
    - использовать большой эмбедер и вообще не делать переранжирование. Опыт показал, что так тоже можно получить желаемое качество. Да на построение индекса требуется больше времени, зато инференс будет очень быстрым
    - работать над качеством эмбедера поменьше, например растить chunk_size и overlap, чтобы эмбедер мог получать больше контекста
  - если не получается без реранкера, то нужно выбирать модель поменьше и найти оптимальное небольшое кол-во кандидатов для переранжирования
</div>

# Проект модуля. Retrieval-система по статьям с arXiv
В проекте вам предстоит построить retrieval-систему и оценить её качество. У вас будет датасет с аннотациями статей с портала arXiv.org и тестовый набор запросов и правильных ответов к ним.

## Проект. Retrieval-система по статьям с arXiv
### Цель
Вам нужно построить retrieval-систему и оценить её качество на тестовых запросах, а также выполнить профилирование частей модели.

### Целевая метрика
Целевая метрика — качество поисковой выдачи: Mean Reciprocal Rank (MRR@5). Проект считается выполненным при `MRR@5 > 0.91`.

### Датасет
Скачайте [архив с датасетом](https://disk.yandex.ru/d/_hlESjxRVivrdg). В нём есть метаданные статей `arxiv-metadata-s.json` и набор тестовых запросов для оценки качества поиска `test_sample.csv`.
`test_sample.csv` имеет структуру:
- `query` — запрос к поисковой системе,
- `id` — ID статьи, отвечающей на запрос `query` (один запрос — один верный ответ),
- `abstract` — аннотация статьи id.

`arxiv-metadata-s.json` имеет структуру:
- `id` — ID статьи,
- `abstract` — аннотация статьи,
- `title` — название статьи.

# Задание
Выполните проект по этапам. 

## Этап 1. Исследовательский анализ (EDA)
- Загрузите датасет и визуализируйте часть данных. Изучите то, с чем вам предстоит работать.
- Обдумайте:
  - как вы будете решать задачу;
  - какие подходы к поиску примените и почему;
  - какие модели выберете и от каких ограничений будете отталкиваться;
  - из чего будет состоять ваша система.

`Результаты`: EDA и выводы о необходимых компонентах системы в Markdown-ячейке.

## Этап 2. Реализация retrieval-системы
Реализуйте:
- код поисковой системы,
- функцию вычисления целевой метрики,
- функцию профилирования всех компонент системы.

`Результат`: поисковая система, способная принимать текстовые запросы и возвращать наиболее релевантные ответы из предоставленного датасета. В систему загружены статьи с их атрибутами и отображён размер получившейся базы.

## Этап 3. Оценка качества системы
Выполните поиск всех тестовых запросов к системе из этапа 2. Оцените MRR и визуализируйте время работы каждой составной части системы.

`Результат`: выполнен поиск тестовых запросов и получена целевая метрика качества. Отображён временной профиль работы системы.

## Этап 4. Выводы
На основании полученных результатов оцените наиболее узкие с точки зрения производительности места всей системы. Предложите подходы к повышению скорости их работы.

`Результат`: описаны медленные части системы и сформированы предложения по их ускорению.


---
## Приступим
---

## Этап 1. Исследовательский анализ (EDA)
- Загрузите датасет и визуализируйте часть данных. Изучите то, с чем вам предстоит работать.
- Обдумайте:
  - как вы будете решать задачу;
  - какие подходы к поиску примените и почему;
  - какие модели выберете и от каких ограничений будете отталкиваться;
  - из чего будет состоять ваша система.

`Результаты`: EDA и выводы о необходимых компонентах системы в Markdown-ячейке.

In [1]:
import json

with open("nlp_s3_project/arxiv-metadata-s.json") as file:
    metadata = json.load(file)

len(metadata)

98213

Датасет состоит почти из 100K документов

In [2]:
metadata[0]

{'id': '0704.0038',
 'submitter': 'Maxim A. Yurkin',
 'authors': 'Maxim A. Yurkin, Alfons G. Hoekstra',
 'title': 'The discrete dipole approximation: an overview and recent developments',
 'comments': '36 pages, 1 figure; added several corrections according to the\n  published erratum except for Eq.(5) (it was correct in the original paper)',
 'journal-ref': 'J.Quant.Spectrosc.Radiat.Transf. 106, 558-589 (2007); Erratum:\n  J.Quant.Spectrosc.Radiat.Transf. 171, 82-83 (2016)',
 'doi': '10.1016/j.jqsrt.2007.01.034 10.1016/j.jqsrt.2015.11.025',
 'report-no': None,
 'categories': 'physics.optics physics.comp-ph',
 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/',
 'abstract': '  We present a review of the discrete dipole approximation (DDA), which is a\ngeneral method to simulate light scattering by arbitrarily shaped particles. We\nput the method in historical context and discuss recent developments, taking\nthe viewpoint of a general framework based on the integral equation

Для каждой статьи есть довольно много метаданных: название, авторы, выдержка и др. Однако в проекте описаны только 3: id, abstract, title. Попробуем сосредоточиться только на них .

In [3]:
import pandas as pd

test_sample = pd.read_csv('nlp_s3_project/test_sample.csv')
test_sample

,id,abstract,query
0,2412.16732,A new platinate was recently discovered when...,What unique composition and decomposition beha...
1,nucl-th/9602019,The production cross sections of various fra...,How does the inclusion of statistical decay af...
2,2501.05500,This survey provides a comprehensive examina...,What are the core components of modern zero-kn...
3,2506.20892,A critical challenge for operating fusion burn...,How does impurity seeding affect the timing an...
4,2208.02031,"In this work, we present the first corpus fo...",What is the primary challenge of the newly dev...
...,...,...,...
995,2409.18116,We estimate the average of any arithmetic fu...,What are the conditions under which we can est...
996,2311.13142,We focus on a scenario of non-Hermitian bulk...,What new application does the adapted non-Herm...
997,astro-ph/0604183,We carried out a one-night optical V and nea...,What evidence suggests that the near-infrared ...
998,1707.08140,Vertically stacked van der Waals heterostruc...,What effect does processing in an inert gas en...


Посмотрим какие бывают запросы

In [4]:
test_sample["query"].head(10).tolist()

['What unique composition and decomposition behavior does the newly discovered platinate Nd10.67Pt4O24 exhibit under thermal conditions?',
 'How does the inclusion of statistical decay affect the shape of mass and charge distributions in quantum molecular dynamics simulations?',
 'What are the core components of modern zero-knowledge proofs discussed in the context of verifiable computing?',
 'How does impurity seeding affect the timing and magnitude of energy loss during ELM crashes?',
 'What is the primary challenge of the newly developed German ADR detection corpus in terms of its annotation?',
 'Can complex potentials serve as both refractive and absorptive optical devices without altering the energy spectrum?',
 'What properties does the set $A_\\mu$ exhibit under various conditions related to the amenability of a group $G$?',
 'How does the CSDW module improve change detection in deep learning by leveraging both spatial and channel differences?',
 'What is the advantage of using 

Выглядит так, что вопросы покрывают в основном контент статей, но не их метаданные. Нет вопросов про авторов или даты публикаций. Поэтому сосредоточимся на оценке близости запросов с выдержками из статей

И так, нужно решить задачу ранжирования документов по запросу.

Целевая метрика MRR@5, значит на каждый запрос нужно будет предоставить 5 кандидатов, чтобы оценить обратные ранги. В базе порядка 100K документов

Для этого развернем векторную БД для поиска релевантных документов и reranker выбора из них топ 5 самых релевантных.

Воспользуемся библиотекой langchain

---
## Этап 2. Реализация retrieval-системы
Реализуйте:
- код поисковой системы,
- функцию вычисления целевой метрики,
- функцию профилирования всех компонент системы.

Напишем загрузчик документов в формат списка эксземпляров класса Document из langchain

In [5]:
from langchain_community.document_loaders import JSONLoader

def load_arxiv_docs(file_path="nlp_s3_project/arxiv-metadata-s.json"):
    loader = JSONLoader(
        file_path=file_path,
        jq_schema=".[]",
        content_key="abstract",
        metadata_func=lambda record, metadata: {
            "id": record.get("id", ""),
            "title": record.get("title", ""),
        }
    )
    return loader.load()

docs = load_arxiv_docs()
len(docs)

98213

In [6]:
docs[0]

Document(metadata={'id': '0704.0038', 'title': 'The discrete dipole approximation: an overview and recent developments'}, page_content='  We present a review of the discrete dipole approximation (DDA), which is a\ngeneral method to simulate light scattering by arbitrarily shaped particles. We\nput the method in historical context and discuss recent developments, taking\nthe viewpoint of a general framework based on the integral equations for the\nelectric field. We review both the theory of the DDA and its numerical aspects,\nthe latter being of critical importance for any practical application of the\nmethod. Finally, the position of the DDA among other methods of light\nscattering simulation is shown and possible future developments are discussed.\n')

In [7]:
print(docs[0])

page_content='  We present a review of the discrete dipole approximation (DDA), which is a
general method to simulate light scattering by arbitrarily shaped particles. We
put the method in historical context and discuss recent developments, taking
the viewpoint of a general framework based on the integral equations for the
electric field. We review both the theory of the DDA and its numerical aspects,
the latter being of critical importance for any practical application of the
method. Finally, the position of the DDA among other methods of light
scattering simulation is shown and possible future developments are discussed.
' metadata={'id': '0704.0038', 'title': 'The discrete dipole approximation: an overview and recent developments'}


Документы загружены. Заголовок и id статьи загружены в метаданные, а выжимка загружена как контент документа

Займемся профилированием

In [8]:
import time
import math
from contextlib import contextmanager
from collections import defaultdict


class Timeline:
    def __init__(self):
        self.spans = []
        self._t0 = time.perf_counter()

    @contextmanager
    def span(self, name):
        start = time.perf_counter()
        try:
            yield
        finally:
            end = time.perf_counter()
            self.spans.append((name, start - self._t0, end - self._t0))

    def reset(self):
        self.spans.clear()
        self._t0 = time.perf_counter()

    # --- агрегация по имени ---
    def aggregate(self):
        durations = defaultdict(list)
        for name, s, e in self.spans:
            durations[name].append(e - s)
        return durations

    @staticmethod
    def _pct(sorted_vals, p):
        if len(sorted_vals) == 1:
            return sorted_vals[0]
        k = (len(sorted_vals) - 1) * p
        f, c = math.floor(k), math.ceil(k)
        if f == c:
            return sorted_vals[int(k)]
        return sorted_vals[f] * (c - k) + sorted_vals[c] * (k - f)

    def report(self):
        agg = self.aggregate()
        header = (f"{'span':50s} {'n':>6s} {'total':>9s} {'mean':>9s} "
                  f"{'p50':>9s} {'p95':>9s} {'max':>9s}")
        print(header)
        print("-" * len(header))
        # сортируем по суммарному времени — самые «дорогие» стадии сверху
        for name, ds in sorted(agg.items(), key=lambda kv: -sum(kv[1])):
            s = sorted(ds)
            print(f"{name:50s} {len(ds):6d} "
                  f"{sum(ds):8.3f}s {sum(ds)/len(ds):8.3f}s "
                  f"{self._pct(s, .50):8.3f}s {self._pct(s, .95):8.3f}s "
                  f"{s[-1]:8.3f}s")

In [39]:
from typing import List, Optional, Set, Any

import faiss
import os
import gc
import pickle
import numpy as np
from tqdm.auto import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

import torch
from torch import Tensor
import torch.nn.functional as F


tl = Timeline()


class RAG:
    def __init__(
        self,
        embedder_name: str = "Qwen/Qwen3-Embedding-0.6B",
        reranker_name: str = "Qwen/Qwen3-Reranker-0.6B",
        chunk_size: int = 500,
        chunk_overlap: int = 125,
        device: Optional[str] = None,
        max_length: int = 8192,
        embedder_torch_dtype: Optional[torch.dtype] = None,
        embedder_attn: Optional[str] = None,
    ):
        self.device = device or ("cuda"
                                 if torch.cuda.is_available() else "cpu")
        if self.device == "cuda":
            torch.cuda.empty_cache()
            print(f"Выбран девайс: {torch.cuda.get_device_name(0)}")
        self.emb_tokenizer = AutoTokenizer.from_pretrained(embedder_name)
        self.embedder = AutoModel\
            .from_pretrained(embedder_name,
                             torch_dtype=embedder_torch_dtype,
                             attn_implementation=embedder_attn,
                             )\
            .to("cpu")
        self.embedder.eval()

        self.rr_tokenizer = AutoTokenizer.from_pretrained(
            reranker_name,
            padding_side='left')
        self.reranker = AutoModelForCausalLM.from_pretrained(
            reranker_name).to("cpu")
        self.reranker.eval()

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        self.index = None
        self.doc_store = []

        self.max_length = max_length
        self.token_false_id = self.rr_tokenizer.convert_tokens_to_ids("no")
        self.token_true_id = self.rr_tokenizer.convert_tokens_to_ids("yes")
        prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
        suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        self.prefix_tokens = self.rr_tokenizer.encode(prefix,
                                                      add_special_tokens=False)
        self.suffix_tokens = self.rr_tokenizer.encode(suffix,
                                                      add_special_tokens=False)

    def _load_embedder_to_gpu(self):
        self.embedder.to(self.device)

    def _unload_embedder(self):
        self.embedder.to("cpu")

    def _load_reranker_to_gpu(self):
        self.reranker.to(self.device)

    def _unload_reranker(self):
        self.reranker.to("cpu")

    def _generate_embeddings(self, texts: List[str], profile_title: str) -> np.ndarray:
        with tl.span(f"{profile_title}. Tokenizing"):
            inputs = self.emb_tokenizer(
                texts,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=self.max_length,
            ).to(self.device)

        with torch.inference_mode():
            with torch.autocast("cuda"):
                with tl.span(f"{profile_title}. Computing embeddings"):
                    outputs = self.embedder(**inputs)

        with tl.span(f"{profile_title}. Processing embeddings"):
            embeddings = self.last_token_pool(outputs.last_hidden_state,
                                              inputs.attention_mask)
            embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy()

    @staticmethod
    def last_token_pool(last_hidden_states: Tensor,
                        attention_mask: Tensor) -> Tensor:
        left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
        if left_padding:
            return last_hidden_states[:, -1]
        else:
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[
                torch.arange(batch_size, device=last_hidden_states.device),
                sequence_lengths]

    def load_and_process_file(self, file_path: str) -> List[Document]:
        """Загрузка и экстракция данных из файлов"""
        docs = load_arxiv_docs(file_path)
        return self.text_splitter.split_documents(docs)

    def build_index(self, file_paths: List[str], batch_size: int = 32) -> None:
        """Строим индекс FAISS"""
        all_docs = []
        for path in file_paths:
            all_docs.extend(self.load_and_process_file(path))
        self.doc_store = all_docs

        # Инициализируем индекс
        hidden_size = self.embedder.config.hidden_size
        self.index = faiss.IndexFlatL2(hidden_size)

        self._load_embedder_to_gpu()
        try:
            for i in tqdm(range(0, len(all_docs), batch_size), desc="Building index"):
                batch = [doc.page_content for doc in all_docs[i:i + batch_size]]
                self.index.add(self._generate_embeddings(batch, "Building index"))
        finally:
            self._unload_embedder()
            torch.cuda.empty_cache()

    def save_index(self, path: str):
        """Сохраняет FAISS индекс и doc_store."""
        if self.index is None:
            raise ValueError("Index not initialized")
        os.makedirs(path, exist_ok=True)
        faiss.write_index(
            self.index,
            os.path.join(path, "index.faiss")
        )
        with open(os.path.join(path, "doc_store.pkl"), "wb") as f:
            pickle.dump(self.doc_store, f)
        print(f"Index saved to {path}")

    def load_index(self, path: str):
        """Загружает ранее сохранённый индекс."""
        index_path = os.path.join(path, "index.faiss")
        docs_path = os.path.join(path, "doc_store.pkl")
        if not os.path.exists(index_path):
            raise FileNotFoundError(index_path)
        if not os.path.exists(docs_path):
            raise FileNotFoundError(docs_path)
        self.index = faiss.read_index(index_path)
        with open(docs_path, "rb") as f:
            self.doc_store = pickle.load(f)

    @staticmethod
    def get_detailed_instruct(task_description: str, query: str):
        return f'Instruct: {task_description}\nQuery:{query}'

    @staticmethod
    def format_reranker_instruction(query, doc, instruction=None):
        if instruction is None:
            instruction = 'Given a web search query, retrieve relevant passages that answer the query'
        output = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(
            instruction=instruction, query=query, doc=doc)
        return output

    def process_inputs(self, pairs):
        """Обработка данных для реранкера"""
        inputs = self.rr_tokenizer(pairs,
                                   padding=False,
                                   truncation='longest_first',
                                   return_attention_mask=False,
                                   max_length=self.max_length -
                                   len(self.prefix_tokens) -
                                   len(self.suffix_tokens))
        for i, ele in enumerate(inputs['input_ids']):
            inputs['input_ids'][i] = self.prefix_tokens + ele + self.suffix_tokens
        inputs = self.rr_tokenizer.pad(inputs,
                                       padding=True,
                                       return_tensors="pt",
                                       max_length=self.max_length)
        return inputs

    def search(self,
               query: str,
               k: int = 5,
               task: str = None,
               load_to_gpu=True):
        if self.index is None:
            raise ValueError("Index not initialized")

        if task is None:
            task = 'Given a web search query, retrieve relevant passages that answer the query'

        with tl.span("L1. Query embeddings"):
            if load_to_gpu:
                self._load_embedder_to_gpu()
            try:
                query_embedding = self._generate_embeddings([query], "L1")
            finally:
                if load_to_gpu:
                    self._unload_embedder()
        with tl.span("L1. Index search"):
            _, indices = self.index.search(query_embedding, k)
        return indices

    @torch.no_grad()
    def compute_logits(self, inputs):
        for key in inputs:
            inputs[key] = inputs[key].to(self.device)
        batch_scores = self.reranker(**inputs).logits[:, -1, :]
        true_vector = batch_scores[:, self.token_true_id]
        false_vector = batch_scores[:, self.token_false_id]
        batch_scores = torch.stack([false_vector, true_vector], dim=1)
        batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
        scores = batch_scores[:, 1].exp().tolist()
        return scores

    def rerank(self, query: str, documents: List[str], batch_size=1, verbose=False, load_to_gpu=True):
        pairs = []
        for d in documents:
            pairs.append(self.format_reranker_instruction(query, d))

        if load_to_gpu:
            self._load_reranker_to_gpu()
        try:
            scores = []
            for i in tqdm(range(0, len(pairs), batch_size), desc="Reranking", disable=not verbose):
                with tl.span("L2. Processing inputs"):
                    inputs = self.process_inputs(pairs[i:i + batch_size])
                with tl.span("L2. Computing logits"):
                    sc = self.compute_logits(inputs)
                scores.extend(sc)
        finally:
            if load_to_gpu:
                self._unload_reranker()
        return scores

    def search_and_rerank(self, query: str, l1_k: int = 100, l2_k: int = 5,
                          rerank_batch_size: int = 1, verbose=False, load_to_gpu=True):
        l1_indices = self.search(query, l1_k, load_to_gpu=load_to_gpu)
        l1_candidates = [self.doc_store[i].page_content for i in l1_indices[0]]
        scores = self.rerank(query, l1_candidates, rerank_batch_size,
                             verbose=verbose, load_to_gpu=load_to_gpu)
        l2_indices = np.argsort(np.array(scores))[::-1][:l2_k]
        l2_candidates = [self.doc_store[l1_indices[0][i]] for i in l2_indices]
        return l2_candidates
    
    @staticmethod
    def mrr_k(preds: List[Any], targets: List[Set[Any]], k=5):
        """
        preds. Предсказанный рейтинг документов для каждого запроса.
        targets. Множество релевантных документов для каждого запроса.
        """
        total_rr = 0.0
        for pred, target in zip(preds, targets):
            rr = 0.0
            for rank, doc_id in enumerate(pred[:k], start=1):
                if doc_id in target:
                    rr = 1.0 / rank
                    break
            total_rr += rr
        return total_rr / len(preds)
    
    def process_sample(self,
                       sample: pd.DataFrame,
                       l1_k: int = 100,
                       l2_k: int = 5,
                       rerank_batch_size: int = 1,
                       ):
        tqdm.pandas(desc="Processing sample")
        result = sample.copy()
        self._load_reranker_to_gpu()
        result["candidates"] = result.progress_apply(lambda row: self.search_and_rerank(
            row['query'], l1_k, l2_k, rerank_batch_size, load_to_gpu=False,
        ), axis=1)
        self._unload_reranker()
        torch.cuda.empty_cache()
        preds = result["candidates"].apply(lambda candidates:
            [c.metadata["id"] for c in candidates]).tolist()
        targets = result["id"].apply(lambda id: {id}).tolist()
        mrr5 = self.mrr_k(preds, targets, k=5)
        return result, mrr5

Для бейзлайна возьмем маленький энкодер для построения индекса и реранкер по умолчанию (из практики)

In [11]:
rag = RAG(
    embedder_name="sentence-transformers/all-MiniLM-L6-v2",
)

Выбран девайс: Tesla T4


In [12]:
rag.build_index(file_paths=["nlp_s3_project/arxiv-metadata-s.json"])

Building index:   0%|          | 0/8511 [00:00<?, ?it/s]

Маленький энкодер помог быстро составить индекс. Проверим весь пайплайн поиска

In [25]:
query = test_sample.iloc[0]["query"]
rag.search_and_rerank(query, l1_k = 100, l2_k = 5, rerank_batch_size=16, verbose=True)

Reranking:   0%|          | 0/7 [00:00<?, ?it/s]

Reranking: 100%|██████████| 7/7 [00:08<00:00,  1.25s/it]


[Document(metadata={'id': '2412.16732', 'title': 'Crystal structure of Nd10.67Pt4O24, a new neodymium platinate'}, page_content='A new platinate was recently discovered when Nd2O3 was explored as a platinum\ncapture material in the Ostwald process, formed by a direct reaction between\nPtO2(g) and Nd2O3. The crystal structure of this new platinate and its\ncomposition, Nd10.67Pt4O24 , are here reported for the first time. The compound\nis synthesized either by a direct reaction using PtO2(g) or by the citric acid\nchemical route. Based on 3-dimensional electron diffraction data and Rietveld'),
 Document(metadata={'id': '2504.12475', 'title': 'Best practices in Quantum Monte Carlo for metal catalysis: CO hydrolysis on Pt(111)'}, page_content='produce carbon dioxide and hydrogen, with a H-atom dissociated from the formate species and another desorbed at the Pt(111) face.'),
 Document(metadata={'id': '2303.05721', 'title': 'Batch Discovery of New Metal Superhydrides via Chemical Template T

In [24]:
print(test_sample.iloc[0]["query"])
print(test_sample.iloc[0]["id"])

What unique composition and decomposition behavior does the newly discovered platinate Nd10.67Pt4O24 exhibit under thermal conditions?
2412.16732


Отлично! Пайплайн отработал и даже нашел правильный документ и расположил его на первой позиции, т.е. MRR@1=1.

Замерим скорость обработки небольшой выборки запросов

In [30]:
rag_candidates, mrr5 = rag.process_sample(test_sample[:10], l1_k=100, l2_k=5, rerank_batch_size=16)
rag_candidates

Processing sample:   0%|          | 0/10 [00:00<?, ?it/s]

/home/ubuntu/project/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2714: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,id,abstract,query,candidates
0,2412.16732,A new platinate was recently discovered when...,What unique composition and decomposition beha...,[page_content='A new platinate was recently di...
1,nucl-th/9602019,The production cross sections of various fra...,How does the inclusion of statistical decay af...,[page_content='and calculated their different ...
2,2501.05500,This survey provides a comprehensive examina...,What are the core components of modern zero-kn...,[page_content='This survey provides a comprehe...
3,2506.20892,A critical challenge for operating fusion burn...,How does impurity seeding affect the timing an...,[page_content='and poloidal seeding location -...
4,2208.02031,"In this work, we present the first corpus fo...",What is the primary challenge of the newly dev...,[page_content='high topic imbalance make it a ...
5,0902.4052,Complex potentials are constructed as Darbou...,Can complex potentials serve as both refractiv...,[page_content='Resonances are essential for un...
6,1605.04065,"For each symmetric, aperiodic probability me...",What properties does the set $A_\mu$ exhibit u...,"[page_content='For each symmetric, aperiodic p..."
7,2501.10905,Change detection in remote sensing imagery i...,How does the CSDW module improve change detect...,[page_content='Most existing change detection ...
8,2112.00230,We describe a practical algorithm for comput...,What is the advantage of using the new algorit...,[page_content='are demonstrated via simulation...
9,2402.05635,This paper provides a mathematical study of ...,What are the necessary conditions for ensuring...,[page_content='the first rigorous proof of the...


In [33]:
mrr5

0.5

In [34]:
tl.report()

span                                n     total      mean       p50       p95       max
---------------------------------------------------------------------------------------
L2. Computing logits  306  333.087s    1.089s    1.158s    1.507s    1.813s
L1. Index search       44    1.526s    0.035s    0.034s    0.037s    0.040s
L2. Processing inputs  306    1.264s    0.004s    0.004s    0.005s    0.008s
L1. Query embeddings   44    0.258s    0.006s    0.006s    0.007s    0.012s


10 запросов обрабатываются минуту и качество поиска получается с `MRR=0.5`. Получилось долго и хуже целевого значения `MRR=0.91`.

Дольше всего работает этап вычисления логитов реранкера. Попробуем определить насколько хороших кандидатов получает реранкер, можно ли обойтись меньшим количество кандидатов.

In [ ]:
def process_sample_with_l1(rag, sample: pd.DataFrame, l1_k: int = 100):
    tqdm.pandas(desc="Processing sample")
    result = sample.copy()
    result["candidates"] = result.\
        progress_apply(lambda row: rag.search(row['query'], l1_k)[0], axis=1)
    preds = result["candidates"].apply(lambda candidates:
        [rag.doc_store[c].metadata["id"] for c in candidates]).tolist()
    targets = result["id"].apply(lambda id: {id}).tolist()
    return preds, targets


preds, targets = process_sample_with_l1(rag, test_sample, l1_k=100)
for k in range(5, 105, 5):
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"MRR@{k}: {mrrk}")

Processing sample:   0%|          | 0/1000 [00:00<?, ?it/s]

MRR@5: 0.5824499999999997
MRR@10: 0.5913428571428571
MRR@15: 0.5939778055278053
MRR@20: 0.5952925802096078
MRR@25: 0.5959983095527285
MRR@30: 0.5965728818514389
MRR@35: 0.5970585222072878
MRR@40: 0.5971923868411526
MRR@45: 0.5973115136493037
MRR@50: 0.5973736235783818
MRR@55: 0.5974875702536492
MRR@60: 0.5975569003585725
MRR@65: 0.5976050478334535
MRR@70: 0.5976344850628907
MRR@75: 0.5976759874103086
MRR@80: 0.5977396113918123
MRR@85: 0.5977875241743218
MRR@90: 0.5978213654933626
MRR@95: 0.5978534071359304
MRR@100: 0.5979044317944662


Выходит выбранный эмбедер для бейзлайна работает недостаточно хорошо. На первом этапе отбора кандидатов нельзя добиться целевого качество даже если отсмотреть все 100 кандидатов. А нужно добиться высокого качества на 5 кандидатах. Меняем эмбедер

Попробуем эмбедер побольше из практики 

In [42]:
rag = RAG()

Выбран девайс: Tesla T4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

In [44]:
rag.build_index(file_paths=["nlp_s3_project/arxiv-metadata-s.json"], batch_size=128)

Building index:   0%|          | 0/2128 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 262.00 MiB. GPU 0 has a total capacity of 14.58 GiB of which 73.56 MiB is free. Including non-PyTorch memory, this process has 14.50 GiB memory in use. Of the allocated memory 13.12 GiB is allocated by PyTorch, and 1.26 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Эмбедер из практики получается сильно тяжелый для выделенного железа. Не хочется 3 часа строить индекс. Попробую эмбедер полегче

In [47]:
import gc

del rag
gc.collect()
torch.cuda.empty_cache()

In [48]:
rag = RAG(
    embedder_name="BAAI/bge-small-en-v1.5"
)

Выбран девайс: Tesla T4


In [49]:
rag.build_index(file_paths=["nlp_s3_project/arxiv-metadata-s.json"], batch_size=32)

Building index:   0%|          | 0/8511 [00:00<?, ?it/s]

In [50]:
def process_sample_with_l1(rag, sample: pd.DataFrame, l1_k: int = 100):
    tqdm.pandas(desc="Processing sample")
    result = sample.copy()
    result["candidates"] = result.\
        progress_apply(lambda row: rag.search(row['query'], l1_k)[0], axis=1)
    preds = result["candidates"].apply(lambda candidates:
        [rag.doc_store[c].metadata["id"] for c in candidates]).tolist()
    targets = result["id"].apply(lambda id: {id}).tolist()
    return preds, targets


preds, targets = process_sample_with_l1(rag, test_sample, l1_k=100)
for k in range(5, 105, 5):
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"MRR@{k}: {mrrk}")

Processing sample:   0%|          | 0/1000 [00:00<?, ?it/s]

MRR@5: 0.8827000000000004
MRR@10: 0.8860250000000003
MRR@15: 0.886490151515152
MRR@20: 0.8868321621790666
MRR@25: 0.8868776167245211
MRR@30: 0.886985043780224
MRR@35: 0.8870744447018829
MRR@40: 0.8871007604913566
MRR@45: 0.8871944112850074
MRR@50: 0.8871944112850074
MRR@55: 0.8872325376466632
MRR@60: 0.8872325376466632
MRR@65: 0.8872486666789212
MRR@70: 0.8872633725612742
MRR@75: 0.8872633725612742
MRR@80: 0.8872633725612742
MRR@85: 0.8872633725612742
MRR@90: 0.8872633725612742
MRR@95: 0.8872633725612742
MRR@100: 0.8872633725612742


С эмбедером поменьше получилось собрать индекс за 10 минут и качество получилось намного выше, но недостаточное для нашей задачи.

Попробую вернуть Qwen 0.6B, но подкрутить настройки

In [ ]:
del rag
gc.collect()
torch.cuda.empty_cache()

In [13]:
embedder_name = "Qwen/Qwen3-Embedding-0.6B"
rag = RAG(
    embedder_name=embedder_name,
    max_length=512,
    chunk_size=1000,
    chunk_overlap=125,
    embedder_torch_dtype=torch.float16,
    # embedder_attn = "flash_attention_2",
)

Выбран девайс: Tesla T4


In [ ]:
rag.build_index(file_paths=["nlp_s3_project/arxiv-metadata-s.json"], batch_size=32)

Building index:   0%|          | 0/4536 [00:00<?, ?it/s]

Такой индекс собирался целый час

In [ ]:
rag.save_index("faiss_index/Qwen/Qwen3-Embedding-0.6B")

In [14]:
rag.load_index(f"faiss_index/{embedder_name}")

In [17]:
def process_sample_with_l1(rag, sample: pd.DataFrame, l1_k: int = 100):
    tqdm.pandas(desc="Processing sample")
    result = sample.copy()
    rag._load_embedder_to_gpu()
    result["candidates"] = result.\
        progress_apply(lambda row: rag.search(row['query'], l1_k, load_to_gpu=False)[0], axis=1)
    rag._unload_embedder()
    torch.cuda.empty_cache()
    preds = result["candidates"].apply(lambda candidates:
        [rag.doc_store[c].metadata["id"] for c in candidates]).tolist()
    targets = result["id"].apply(lambda id: {id}).tolist()
    return preds, targets


preds, targets = process_sample_with_l1(rag, test_sample, l1_k=100)
for k in range(1, 5):
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"MRR@{k}: {mrrk}")

for k in range(5, 105, 5):
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"MRR@{k}: {mrrk}")

Processing sample:   0%|          | 0/1000 [00:00<?, ?it/s]

MRR@1: 0.91
MRR@2: 0.9285
MRR@3: 0.9338333333333336
MRR@4: 0.9358333333333336
MRR@5: 0.9364333333333338
MRR@10: 0.9372900793650797
MRR@15: 0.9379424103674108
MRR@20: 0.9380012338968224
MRR@25: 0.9380863788243589
MRR@30: 0.9381618773998573
MRR@35: 0.9381618773998573
MRR@40: 0.9381618773998573
MRR@45: 0.9381862676437598
MRR@50: 0.9381862676437598
MRR@55: 0.9381862676437598
MRR@60: 0.9382032167963021
MRR@65: 0.9382032167963021
MRR@70: 0.9382181421694366
MRR@75: 0.9382181421694366
MRR@80: 0.9382181421694366
MRR@85: 0.9382181421694366
MRR@90: 0.9382296364223102
MRR@95: 0.9382296364223102
MRR@100: 0.9382296364223102


С "большим" эмбедером в 0.6B параметров получилось добиться целевого `MRR@5=0.91` даже на первой стадии генерации кандидатов. Получилось добавиться даже `MRR@1=0.91`. Единственный минус индекс строился целый час.

Попробую взять эмбедер полегче, чтобы реализовать полный цикл с двумя стадиями с целевым качеством.

---

И тут я понял, что MRR@100 ниже 0.91 не говорит, что с реранкером не получится получить MRR@5>=0.91, ведь в MRR учитывается позиция. Нужна метрика чтобы оценить наличие правильного документа в выдаче из k кандидатов первого уровня поиска.  

In [17]:
embedder_name = "BAAI/bge-base-en-v1.5"
rag = RAG(
    embedder_name=embedder_name,
    max_length=512,
    chunk_size=1500,
    chunk_overlap=100,
    embedder_torch_dtype=torch.float16,
    # embedder_attn = "flash_attention_2",
)

Выбран девайс: Tesla T4


In [12]:
rag.build_index(file_paths=["nlp_s3_project/arxiv-metadata-s.json"], batch_size=256)

Building index:   0%|          | 0/437 [00:00<?, ?it/s]

In [26]:
rag.save_index(f"faiss_index/{embedder_name}")

Index saved to faiss_index/BAAI/bge-base-en-v1.5


In [19]:
rag.load_index(f"faiss_index/{embedder_name}")

In [ ]:
def process_sample_with_l1(rag, sample: pd.DataFrame, l1_k: int = 100):
    tqdm.pandas(desc="Processing sample")
    result = sample.copy()
    rag._load_embedder_to_gpu()
    result["candidates"] = result.\
        progress_apply(lambda row: rag.search(row['query'], l1_k, load_to_gpu=False)[0], axis=1)
    rag._unload_embedder()
    torch.cuda.empty_cache()
    preds = result["candidates"].apply(lambda candidates:
        [rag.doc_store[c].metadata["id"] for c in candidates]).tolist()
    targets = result["id"].apply(lambda id: {id}).tolist()
    return preds, targets

def coverage(preds: List[Any], targets: List[Set[Any]], k=5):
    total_score = 0.0
    for pred, target in zip(preds, targets):
        score = int(max([doc in target for doc in pred[:k]]))
        total_score += score
    return total_score / len(preds)


preds, targets = process_sample_with_l1(rag, test_sample, l1_k=100)
for k in range(1, 5):
    coveragek = coverage(preds, targets, k=k)
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"Coverage@{k}: {coveragek}   | MRR@{k}: {mrrk}")

for k in range(5, 105, 5):
    coveragek = coverage(preds, targets, k=k)
    mrrk = rag.mrr_k(preds, targets, k=k)
    print(f"Coverage@{k}: {coveragek}   | MRR@{k}: {mrrk}")

Processing sample:   0%|          | 0/1000 [00:00<?, ?it/s]

Coverage@1: 0.859   | MRR@1: 0.859
Coverage@2: 0.905   | MRR@2: 0.882
Coverage@3: 0.923   | MRR@3: 0.8880000000000001
Coverage@4: 0.932   | MRR@4: 0.8902500000000001
Coverage@5: 0.938   | MRR@5: 0.8914500000000002
Coverage@10: 0.958   | MRR@10: 0.8941869047619048
Coverage@15: 0.963   | MRR@15: 0.8945983266733268
Coverage@20: 0.969   | MRR@20: 0.894946937784438
Coverage@25: 0.973   | MRR@25: 0.8951271325896327
Coverage@30: 0.975   | MRR@30: 0.8951960981068741
Coverage@35: 0.977   | MRR@35: 0.8952596061713902
Coverage@40: 0.979   | MRR@40: 0.8953122377503375
Coverage@45: 0.98   | MRR@45: 0.89533662799424
Coverage@50: 0.983   | MRR@50: 0.895398702824172
Coverage@55: 0.984   | MRR@55: 0.8954175707487003
Coverage@60: 0.984   | MRR@60: 0.8954175707487003
Coverage@65: 0.985   | MRR@65: 0.8954339641913233
Coverage@70: 0.985   | MRR@70: 0.8954339641913233
Coverage@75: 0.985   | MRR@75: 0.8954339641913233
Coverage@80: 0.986   | MRR@80: 0.8954464641913232
Coverage@85: 0.987   | MRR@85: 0.89545880

Видно что текущие эмбединги позволяют получить достаточно высокое покрытие правильными кандидатами уже для 3 кандидатов (если реранкер поставит всех на первую позицию, то получим целевое качество).

Т.к. у нас целевая метрика MRR@5, попробуем доставать с первого этапа 5 кандидатов и переранжировать их реранкером

In [33]:
def process_sample(
    rag,
    sample: pd.DataFrame,
    l1_k: int = 100,
    l2_k: int = 5,
    rerank_batch_size: int = 16,
):
    result = sample.copy()
    rag._load_embedder_to_gpu()
    try:
        l1_indices_all = []
        for query in tqdm(result["query"], desc="L1 Search"):
            query_embedding = rag._generate_embeddings([query], "L1")
            _, indices = rag.index.search(query_embedding, l1_k)
            l1_indices_all.append(indices[0])
    finally:
        rag._unload_embedder()
        torch.cuda.empty_cache()

    l1_candidates_all = []
    for indices in l1_indices_all:
        candidates = [rag.doc_store[i] for i in indices]
        l1_candidates_all.append(candidates)

    result["candidates_l1"] = l1_candidates_all

    rag._load_reranker_to_gpu()
    final_candidates = []
    try:
        for query, docs in tqdm(
            zip(result["query"], l1_candidates_all),
            total=len(result),
            desc="L2 Rerank",
        ):
            doc_texts = [d.page_content for d in docs]
            scores = rag.rerank(
                query=query,
                documents=doc_texts,
                batch_size=rerank_batch_size,
                load_to_gpu=False,
            )
            best_idx = np.argsort(scores)[::-1][:l2_k]
            final_candidates.append([docs[i] for i in best_idx])
    finally:
        rag._unload_reranker()
        torch.cuda.empty_cache()

    result["candidates"] = final_candidates
    preds = result["candidates"].\
        apply(lambda candidates: [c.metadata["id"] for c in candidates]).\
        tolist()
    targets = result["id"].apply(lambda x: {x}).tolist()

    mrr5 = rag.mrr_k(preds, targets, k=5)
    return result, mrr5

rag_candidates, mrr5 = process_sample(rag, test_sample, l1_k=5, l2_k=5, rerank_batch_size=16)
rag_candidates

L1 Search:   0%|          | 0/1000 [00:00<?, ?it/s]

L2 Rerank:   0%|          | 0/1000 [00:00<?, ?it/s]

/home/ubuntu/project/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2714: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,id,abstract,query,candidates_l1,candidates
0,2412.16732,A new platinate was recently discovered when...,What unique composition and decomposition beha...,[page_content='A new platinate was recently di...,[page_content='A new platinate was recently di...
1,nucl-th/9602019,The production cross sections of various fra...,How does the inclusion of statistical decay af...,[page_content='The production cross sections o...,[page_content='The production cross sections o...
2,2501.05500,This survey provides a comprehensive examina...,What are the core components of modern zero-kn...,[page_content='This survey provides a comprehe...,[page_content='This survey provides a comprehe...
3,2506.20892,A critical challenge for operating fusion burn...,How does impurity seeding affect the timing an...,[page_content='A critical challenge for operat...,[page_content='A critical challenge for operat...
4,2208.02031,"In this work, we present the first corpus fo...",What is the primary challenge of the newly dev...,"[page_content='In this work, we present the fi...","[page_content='In this work, we present the fi..."
...,...,...,...,...,...
995,2409.18116,We estimate the average of any arithmetic fu...,What are the conditions under which we can est...,[page_content='We estimate the average of any ...,[page_content='We estimate the average of any ...
996,2311.13142,We focus on a scenario of non-Hermitian bulk...,What new application does the adapted non-Herm...,[page_content='We focus on a scenario of non-H...,[page_content='We focus on a scenario of non-H...
997,astro-ph/0604183,We carried out a one-night optical V and nea...,What evidence suggests that the near-infrared ...,[page_content='We carried out a one-night opti...,[page_content='We carried out a one-night opti...
998,1707.08140,Vertically stacked van der Waals heterostruc...,What effect does processing in an inert gas en...,[page_content='pristine interfaces by processi...,[page_content='pristine interfaces by processi...


In [35]:
mrr5

0.9303333333333332

In [36]:
tl.report()

span                                n     total      mean       p50       p95       max
---------------------------------------------------------------------------------------
L2. Computing logits 1010  683.311s    0.677s    0.669s    0.905s    2.680s
L1. Computing embeddings 1011    7.606s    0.008s    0.007s    0.008s    0.015s
L2. Processing inputs 1010    3.310s    0.003s    0.003s    0.004s    0.033s
L1. Tokenizing       1011    0.762s    0.001s    0.001s    0.001s    0.002s
L1. Processing embeddings 1010    0.231s    0.000s    0.000s    0.000s    0.000s
L1. Query embeddings    1    0.001s    0.001s    0.001s    0.001s    0.001s


Получилось добится целевого качества, но скорость поиска низкая за счёт этапа реранкинга. Попробуем реранкер попроще

In [26]:
from typing import List, Optional, Set, Any

import faiss
import os
import gc
import pickle
import numpy as np
from tqdm.auto import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM,\
    AutoModelForSequenceClassification


import torch
from torch import Tensor
import torch.nn.functional as F


tl = Timeline()


class RAG:
    def __init__(
        self,
        embedder_name: str = "Qwen/Qwen3-Embedding-0.6B",
        reranker_name: str = "BAAI/bge-reranker-base",
        chunk_size: int = 500,
        chunk_overlap: int = 125,
        device: Optional[str] = None,
        max_length: int = 8192,
        embedder_torch_dtype: Optional[torch.dtype] = None,
        embedder_attn: Optional[str] = None,
    ):
        self.device = device or ("cuda"
                                 if torch.cuda.is_available() else "cpu")
        if self.device == "cuda":
            torch.cuda.empty_cache()
            print(f"Выбран девайс: {torch.cuda.get_device_name(0)}")
        self.emb_tokenizer = AutoTokenizer.from_pretrained(embedder_name)
        self.embedder = AutoModel\
            .from_pretrained(embedder_name,
                             torch_dtype=embedder_torch_dtype,
                             attn_implementation=embedder_attn,
                             )\
            .to("cpu")
        self.embedder.eval()

        self.rr_tokenizer = AutoTokenizer.from_pretrained(
            reranker_name,
            padding_side='left')
        self.reranker = AutoModelForSequenceClassification.from_pretrained(
            reranker_name).to("cpu")
        self.reranker.eval()

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        self.index = None
        self.doc_store = []

        self.max_length = max_length
        self.token_false_id = self.rr_tokenizer.convert_tokens_to_ids("no")
        self.token_true_id = self.rr_tokenizer.convert_tokens_to_ids("yes")
        prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
        suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        self.prefix_tokens = self.rr_tokenizer.encode(prefix,
                                                      add_special_tokens=False)
        self.suffix_tokens = self.rr_tokenizer.encode(suffix,
                                                      add_special_tokens=False)

    def _load_embedder_to_gpu(self):
        self.embedder.to(self.device)

    def _unload_embedder(self):
        self.embedder.to("cpu")

    def _load_reranker_to_gpu(self):
        self.reranker.to(self.device)

    def _unload_reranker(self):
        self.reranker.to("cpu")

    def _generate_embeddings(self, texts: List[str], profile_title: str) -> np.ndarray:
        with tl.span(f"{profile_title}. Tokenizing"):
            inputs = self.emb_tokenizer(
                texts,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=self.max_length,
            ).to(self.device)

        with torch.inference_mode():
            with torch.autocast("cuda"):
                with tl.span(f"{profile_title}. Computing embeddings"):
                    outputs = self.embedder(**inputs)

        with tl.span(f"{profile_title}. Processing embeddings"):
            embeddings = self.last_token_pool(outputs.last_hidden_state,
                                              inputs.attention_mask)
            embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy()

    @staticmethod
    def last_token_pool(last_hidden_states: Tensor,
                        attention_mask: Tensor) -> Tensor:
        left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
        if left_padding:
            return last_hidden_states[:, -1]
        else:
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[
                torch.arange(batch_size, device=last_hidden_states.device),
                sequence_lengths]

    def load_and_process_file(self, file_path: str) -> List[Document]:
        """Загрузка и экстракция данных из файлов"""
        docs = load_arxiv_docs(file_path)
        return self.text_splitter.split_documents(docs)

    def build_index(self, file_paths: List[str], batch_size: int = 32) -> None:
        """Строим индекс FAISS"""
        all_docs = []
        for path in file_paths:
            all_docs.extend(self.load_and_process_file(path))
        self.doc_store = all_docs

        # Инициализируем индекс
        hidden_size = self.embedder.config.hidden_size
        self.index = faiss.IndexFlatL2(hidden_size)

        self._load_embedder_to_gpu()
        try:
            for i in tqdm(range(0, len(all_docs), batch_size), desc="Building index"):
                batch = [doc.page_content for doc in all_docs[i:i + batch_size]]
                self.index.add(self._generate_embeddings(batch, "Building index"))
        finally:
            self._unload_embedder()
            torch.cuda.empty_cache()

    def save_index(self, path: str):
        """Сохраняет FAISS индекс и doc_store."""
        if self.index is None:
            raise ValueError("Index not initialized")
        os.makedirs(path, exist_ok=True)
        faiss.write_index(
            self.index,
            os.path.join(path, "index.faiss")
        )
        with open(os.path.join(path, "doc_store.pkl"), "wb") as f:
            pickle.dump(self.doc_store, f)
        print(f"Index saved to {path}")

    def load_index(self, path: str):
        """Загружает ранее сохранённый индекс."""
        index_path = os.path.join(path, "index.faiss")
        docs_path = os.path.join(path, "doc_store.pkl")
        if not os.path.exists(index_path):
            raise FileNotFoundError(index_path)
        if not os.path.exists(docs_path):
            raise FileNotFoundError(docs_path)
        self.index = faiss.read_index(index_path)
        with open(docs_path, "rb") as f:
            self.doc_store = pickle.load(f)

    @staticmethod
    def get_detailed_instruct(task_description: str, query: str):
        return f'Instruct: {task_description}\nQuery:{query}'

    def process_inputs(self, pairs):
        """Обработка данных для реранкера"""
        inputs = self.rr_tokenizer(pairs,
                                   padding=False,
                                   truncation='longest_first',
                                   return_attention_mask=False,
                                   max_length=self.max_length -
                                   len(self.prefix_tokens) -
                                   len(self.suffix_tokens))
        for i, ele in enumerate(inputs['input_ids']):
            inputs['input_ids'][i] = self.prefix_tokens + ele + self.suffix_tokens
        inputs = self.rr_tokenizer.pad(inputs,
                                       padding=True,
                                       return_tensors="pt",
                                       max_length=self.max_length)
        return inputs

    def search(self,
               query: str,
               k: int = 5,
               task: str = None,
               load_to_gpu=True):
        if self.index is None:
            raise ValueError("Index not initialized")

        if task is None:
            task = 'Given a web search query, retrieve relevant passages that answer the query'

        with tl.span("L1. Query embeddings"):
            if load_to_gpu:
                self._load_embedder_to_gpu()
            try:
                query_embedding = self._generate_embeddings([query], "L1")
            finally:
                if load_to_gpu:
                    self._unload_embedder()
        with tl.span("L1. Index search"):
            _, indices = self.index.search(query_embedding, k)
        return indices

    def rerank(self, query: str, documents: List[str], batch_size=1, verbose=False, load_to_gpu=True):
        if load_to_gpu:
            self._load_reranker_to_gpu()
        try:
            scores = []
            for i in tqdm(range(0, len(documents), batch_size), desc="Reranking", disable=not verbose):
                docs = documents[i:i + batch_size]
                queries = [query] * len(docs)
                with tl.span("L2. Tokenization"):
                    inputs = self.rr_tokenizer(queries, docs, padding=True, truncation=True, return_tensors="pt")
                    for key in inputs:
                        inputs[key] = inputs[key].to(self.device)
                with tl.span("L2. Reranking"):
                    with torch.no_grad():
                        sc = self.reranker(**inputs).logits.cpu().numpy()
                scores.extend(sc.squeeze(-1))
        finally:
            if load_to_gpu:
                self._unload_reranker()
        return scores

    def search_and_rerank(self, query: str, l1_k: int = 100, l2_k: int = 5,
                          rerank_batch_size: int = 1, verbose=False, load_to_gpu=True):
        l1_indices = self.search(query, l1_k, load_to_gpu=load_to_gpu)
        l1_candidates = [self.doc_store[i].page_content for i in l1_indices[0]]
        scores = self.rerank(query, l1_candidates, rerank_batch_size,
                             verbose=verbose, load_to_gpu=load_to_gpu)
        l2_indices = np.argsort(np.array(scores))[::-1][:l2_k]
        l2_candidates = [self.doc_store[l1_indices[0][i]] for i in l2_indices]
        return l2_candidates
    
    @staticmethod
    def mrr_k(preds: List[Any], targets: List[Set[Any]], k=5):
        """
        preds. Предсказанный рейтинг документов для каждого запроса.
        targets. Множество релевантных документов для каждого запроса.
        """
        total_rr = 0.0
        for pred, target in zip(preds, targets):
            rr = 0.0
            for rank, doc_id in enumerate(pred[:k], start=1):
                if doc_id in target:
                    rr = 1.0 / rank
                    break
            total_rr += rr
        return total_rr / len(preds)
    
    def process_sample(self,
                       sample: pd.DataFrame,
                       l1_k: int = 100,
                       l2_k: int = 5,
                       rerank_batch_size: int = 1,
                       ):
        tqdm.pandas(desc="Processing sample")
        result = sample.copy()
        self._load_reranker_to_gpu()
        result["candidates"] = result.progress_apply(lambda row: self.search_and_rerank(
            row['query'], l1_k, l2_k, rerank_batch_size, load_to_gpu=False,
        ), axis=1)
        self._unload_reranker()
        torch.cuda.empty_cache()
        preds = result["candidates"].apply(lambda candidates:
            [c.metadata["id"] for c in candidates]).tolist()
        targets = result["id"].apply(lambda id: {id}).tolist()
        mrr5 = self.mrr_k(preds, targets, k=5)
        return result, mrr5

In [27]:
embedder_name = "BAAI/bge-base-en-v1.5"
reranker_name = "BAAI/bge-reranker-base"

rag = RAG(
    embedder_name=embedder_name,
    reranker_name=reranker_name,
    max_length=512,
    chunk_size=1500,
    chunk_overlap=100,
    embedder_torch_dtype=torch.float16,
    # embedder_attn = "flash_attention_2",
)

Выбран девайс: Tesla T4


In [28]:
rag.load_index(f"faiss_index/{embedder_name}")

In [31]:
def process_sample(
    rag,
    sample: pd.DataFrame,
    l1_k: int = 100,
    l2_k: int = 5,
    rerank_batch_size: int = 16,
):
    result = sample.copy()
    rag._load_embedder_to_gpu()
    try:
        l1_indices_all = []
        for query in tqdm(result["query"], desc="L1 Search"):
            query_embedding = rag._generate_embeddings([query], "L1")
            _, indices = rag.index.search(query_embedding, l1_k)
            l1_indices_all.append(indices[0])
    finally:
        rag._unload_embedder()
        torch.cuda.empty_cache()

    l1_candidates_all = []
    for indices in l1_indices_all:
        candidates = [rag.doc_store[i] for i in indices]
        l1_candidates_all.append(candidates)

    result["candidates_l1"] = l1_candidates_all

    rag._load_reranker_to_gpu()
    final_candidates = []
    try:
        for query, docs in tqdm(
            zip(result["query"], l1_candidates_all),
            total=len(result),
            desc="L2 Rerank",
        ):
            doc_texts = [d.page_content for d in docs]
            scores = rag.rerank(
                query=query,
                documents=doc_texts,
                batch_size=rerank_batch_size,
                load_to_gpu=False,
            )
            best_idx = np.argsort(scores)[::-1][:l2_k]
            final_candidates.append([docs[i] for i in best_idx])
    finally:
        rag._unload_reranker()
        torch.cuda.empty_cache()

    result["candidates"] = final_candidates
    preds = result["candidates"].\
        apply(lambda candidates: [c.metadata["id"] for c in candidates]).\
        tolist()
    targets = result["id"].apply(lambda x: {x}).tolist()

    mrr5 = rag.mrr_k(preds, targets, k=5)
    return result, mrr5

rag_candidates, mrr5 = process_sample(rag, test_sample, l1_k=5, l2_k=5, rerank_batch_size=16)
rag_candidates

L1 Search:   0%|          | 0/1000 [00:00<?, ?it/s]

L2 Rerank:   0%|          | 0/1000 [00:00<?, ?it/s]

,id,abstract,query,candidates_l1,candidates
0,2412.16732,A new platinate was recently discovered when...,What unique composition and decomposition beha...,[page_content='A new platinate was recently di...,[page_content='A new platinate was recently di...
1,nucl-th/9602019,The production cross sections of various fra...,How does the inclusion of statistical decay af...,[page_content='The production cross sections o...,[page_content='The production cross sections o...
2,2501.05500,This survey provides a comprehensive examina...,What are the core components of modern zero-kn...,[page_content='This survey provides a comprehe...,[page_content='This survey provides a comprehe...
3,2506.20892,A critical challenge for operating fusion burn...,How does impurity seeding affect the timing an...,[page_content='A critical challenge for operat...,[page_content='A critical challenge for operat...
4,2208.02031,"In this work, we present the first corpus fo...",What is the primary challenge of the newly dev...,"[page_content='In this work, we present the fi...","[page_content='In this work, we present the fi..."
...,...,...,...,...,...
995,2409.18116,We estimate the average of any arithmetic fu...,What are the conditions under which we can est...,[page_content='We estimate the average of any ...,[page_content='We estimate the average of any ...
996,2311.13142,We focus on a scenario of non-Hermitian bulk...,What new application does the adapted non-Herm...,[page_content='We focus on a scenario of non-H...,[page_content='We focus on a scenario of non-H...
997,astro-ph/0604183,We carried out a one-night optical V and nea...,What evidence suggests that the near-infrared ...,[page_content='We carried out a one-night opti...,[page_content='We carried out a one-night opti...
998,1707.08140,Vertically stacked van der Waals heterostruc...,What effect does processing in an inert gas en...,[page_content='pristine interfaces by processi...,[page_content='Vertically stacked van der Waal...


In [32]:
mrr5

0.9192

In [34]:
tl.report()

span                                                    n     total      mean       p50       p95       max
-----------------------------------------------------------------------------------------------------------
L2. Reranking                                        1010   82.047s    0.081s    0.086s    0.114s    0.141s
L1. Computing embeddings                             1010    7.730s    0.008s    0.008s    0.008s    0.016s
L2. Tokenization                                     1010    3.108s    0.003s    0.003s    0.004s    0.025s
L1. Tokenizing                                       1010    0.761s    0.001s    0.001s    0.001s    0.002s
L1. Processing embeddings                            1010    0.239s    0.000s    0.000s    0.000s    0.000s


Получилось добиться целевого качества поиска `MRR@5=0.9192` используя довольно легкие ембедер и реранкер. Обработка 1000 тестовых запросов заняла около 2 минут.